In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import base64

load_dotenv()

True

In [2]:
# Encode the image and sent to vision model

In [4]:
with open ("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

image_b64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAmwAAAHgCAIAAACXbaZMAACxv0lEQVR4nOzdeVwT1944/iGBkASIIKuyBBAQIahYqiBuiCjWrVi8eK+KYlVEqdvFBQtq64obVvsgUhatD9XSihuLFatYZRNUtA0iYF0AZZEiEBICSeb3ejy/O9+52QiRzfp5/5WcOXPmzDnDfJiZkzka'

In [6]:
llm = ChatGroq(model = "qwen/qwen3.8-27b")

message =  HumanMessage(content = [
    {"type":"image_url", "image_url":{"url": f"data:image/png;base64,{image_b64}"}},
    {"type":"text",  "text":"This is a blood work report. Extract all the test results and flage any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)

Based on the blood work report provided, here are the extracted test results. I have flagged any values that fall outside the stated normal ranges.

### **Patient Information**
*   **Patient:** Rajesh Sharma
*   **Age/Sex:** 48, Male
*   **Date:** May 7, 2026

---

### **Complete Blood Count (CBC)**
*   **Hemoglobin:** 15.1 g/dL (Normal: 13.5–17.5)
*   **Hematocrit:** 44% (Normal: 41–53%)
*   **WBC:** 6.8 x10^3/uL (Normal: 4.5–11.0)
*   **Platelets:** 220 x10^3/uL (Normal: 150–400)

**Status:** All values within normal range.

---

### **Lipid Panel**
*   **Total Cholesterol:** 238 mg/dL (Normal: <200) — **HIGH** ⚠️
*   **LDL Cholesterol:** 162 mg/dL (Normal: <100) — **HIGH** ⚠️
*   **HDL Cholesterol:** 36 mg/dL (Normal: >40) — **LOW** ⚠️
*   **Triglycerides:** 188 mg/dL (Normal: <150) — **HIGH** ⚠️

**Status:** All values are outside the normal range, indicating a lipid profile that is typically associated with increased cardiovascular risk.

---

### **Metabolic Panel**
*   **Glucose

In [13]:
from langchain.tools import tool

@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])


In [14]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [15]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

print(result["messages"][-1].content)

### Blood Work Analysis
- **Complete Blood Count (CBC):**
  - Hemoglobin: 15.1 g/dL (Normal: 13.5–17.5)
  - Hematocrit: 44% (Normal: 41–53%)
  - WBC: 6.8 x 10^3/uL (Normal: 4.5–11.0)
  - Platelets: 220 x 10^3/uL (Normal: 150–400)  
  **Interpretation:** CBC values are within normal ranges.

- **Lipid Panel:**
  - Total Cholesterol: 238 mg/dL (Normal: <200) → **High**
  - LDL Cholesterol: 162 mg/dL (Normal: <100) → **High**
  - HDL Cholesterol: 36 mg/dL (Normal: >40) → **Low**
  - Triglycerides: 188 mg/dL (Normal: <150) → **High**  
  **Interpretation:** Abnormal lipid profile indicating elevated cholesterol and triglycerides.

- **Metabolic Panel:**
  - Glucose (Fasting): 92 mg/dL (Normal: 70–99)
  - HbA1c: 5.3% (Normal: <5.7%)
  - Creatinine: 1.0 mg/dL (Normal: 0.7–1.3)
  - eGFR: 82 mL/min (Normal: >60)  
  **Interpretation:** Metabolic values are within normal ranges.

### Condition Categorization
Based on the elevated total cholesterol, LDL cholesterol, and triglycerides, the condit